# Day 54 — Monitoring & model governance
Objectives:
- Monitor data quality and drift.
- Track model performance post-deployment.
- Governance basics: model registry, approvals, audit trail.
Note: This notebook simulates drift and basic monitoring metrics.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(42)
# Simulate baseline score distribution
y_true = rng.integers(0,2, size=5000)
y_score = y_true*0.7 + (1-y_true)*0.3 + rng.normal(0,0.1, size=5000)
baseline_auc = roc_auc_score(y_true, y_score)
baseline_auc


In [ ]:
# Simulate drift: feature shift lowers separability
y_true2 = rng.integers(0,2, size=5000)
y_score2 = y_true2*0.6 + (1-y_true2)*0.4 + rng.normal(0,0.15, size=5000)
live_auc = roc_auc_score(y_true2, y_score2)
delta = live_auc - baseline_auc
baseline_auc, live_auc, delta


## Data drift detection (simple)
- Monitor feature means/stds, PSI (Population Stability Index) for scoring bands.
Below is a simple PSI demo for score bins.

In [ ]:
def psi(expected, actual, bins=10):
    e_perc, _ = np.histogram(expected, bins=bins, range=(expected.min(), expected.max()))
    a_perc, _ = np.histogram(actual,   bins=bins, range=(expected.min(), expected.max()))
    e_perc = e_perc / e_perc.sum(); a_perc = a_perc / a_perc.sum()
    # add tiny value to avoid div/0
    e_perc = np.clip(e_perc, 1e-6, None); a_perc = np.clip(a_perc, 1e-6, None)
    return np.sum((a_perc - e_perc) * np.log(a_perc / e_perc))
psi_value = psi(y_score, y_score2)
psi_value


## Governance checklist
- Register models in a Model Registry (MLflow, SageMaker, etc.).
- Record: training data snapshot/hash, code version (git sha), hyperparams, metrics.
- Approval workflow for promotion to production (staging → prod).
- Periodic re-validation; drift + performance alerts.

## Exercises
1) Compute PSI for multiple features or scores over time windows.
2) Create a simple dashboard (pandas+matplotlib) showing AUC and PSI trends weekly.
3) Draft a governance policy document: roles, approvals, rollbacks.